# Batch Preprocessing

Run the finalized resting-state EEG preprocessing pipeline across downloaded subjects.


## Setup


In [ ]:
from importlib import reload
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

import src.eeg_preprocessing as eeg_preprocessing

reload(eeg_preprocessing)

project_root = Path.cwd()
if not (project_root / "data" / "ds004796").exists():
    project_root = project_root.parent

data_root = project_root / "data" / "ds004796"
output_dir = project_root / "data" / "processed" / "eeg"

print(f"Project root: {project_root}")
print(f"Data root: {data_root}")
print(f"Output dir: {output_dir}")


## Find Subjects


In [ ]:
vhdr_paths = eeg_preprocessing.find_rest_vhdrs(data_root)
subject_ids = [eeg_preprocessing.subject_id_from_vhdr(path) for path in vhdr_paths]

print(f"Found {len(subject_ids)} rest EEG files")
subject_ids


## Run Preprocessing


In [ ]:
results, run_qc = eeg_preprocessing.preprocess_all_rest_subjects(
    data_root=data_root,
    output_dir=output_dir,
    subject_ids=None,
    continue_on_error=True,
    use_icalabel=True,
    ica_n_components=None,
)

run_qc


## Review QC Outputs


In [ ]:
prep_qc = pd.read_csv(output_dir / "rest_preprocessing_prep_qc.tsv", sep="\t")
ica_qc = pd.read_csv(output_dir / "rest_preprocessing_ica_qc.tsv", sep="\t")
epoch_qc = pd.read_csv(output_dir / "rest_preprocessing_epoch_qc.tsv", sep="\t")

qc_tables = {
    "Overall QC": run_qc,
    "PREP QC": prep_qc,
    "ICA QC": ica_qc,
    "Epoch QC": epoch_qc,
}

for title, table in qc_tables.items():
    display(Markdown(f"### {title}"))
    display(table)


## Flag Subjects For Review


In [ ]:
display(Markdown("### Subjects with preprocessing errors"))
display(run_qc.loc[run_qc["status"] != "ok"])

display(Markdown("### Most still-noisy channels after PREP"))
display(
    prep_qc.sort_values("n_still_noisy_channels", ascending=False)[
        ["subject_id", "condition", "n_still_noisy_channels", "still_noisy_channels"]
    ]
)

display(Markdown("### Most rejected epochs"))
display(
    epoch_qc.sort_values("percent_epochs_dropped", ascending=False)[
        [
            "subject_id",
            "condition",
            "n_epochs_before_rejection",
            "n_epochs_after_rejection",
            "percent_epochs_dropped",
            "excluded_bad_channels",
        ]
    ]
)
